---
# README — UK Fuel Poverty Risk Scoring Project (Python edition, Stages 3-7)
---

This notebook is a Python conversion of `Fuel_Poverty.ipynb` (originally R).
**Stages 1-2 (Steps 01-06: raw data ingestion and master dataset construction)
have been removed** — their outputs are already computed, and the pipeline now
starts from Stage 3 using the saved Stage 2 output
(`clean_master_combined.parquet`). All downstream logic is unchanged; only the
language and libraries differ (see the library mapping in the pipeline
overview below).

---
### DIRECTORY STRUCTURE
*Set up your project folder as follows (Stage 1 raw files are no longer needed):*

```
main/
   └── raw_files/
       │
       ├── -- STAGE 2 OUTPUT (starting input, produced by the original Steps 01-06) --
       ├── clean_master_combined.parquet
       │
       ├── -- FUEL POVERTY PRIOR / LILEE DATA (used by Step 09) --
       ├── LILEE_supplementary_tables_2024.xlsx
       └── mcmc_probabilities.xlsx
```
---
### IMPORTANT NOTES BEFORE RUNNING
---

1. HARD-CODED PATHS: `ROOT_PATH` in CELL 0 points at the original machine's
   working directory. Update it to your `main/raw_files/` directory before
   running — everything else derives from it automatically.

2. INTERMEDIATE FILES: Several cells depend on outputs produced by earlier
   cells (e.g., `clean_master_final.parquet`, `mcmc_split_clean_master.parquet`,
   `ml_model_data_with_predictions.parquet`). These are created during the run
   and saved to `ROOT_PATH`. Do not delete them between steps.

3. DNN ENVIRONMENT: the original R notebook required a "batnet" conda
   environment via reticulate. The Python version only needs `tensorflow`
   installed in the notebook kernel's environment.

4. MEMORY: the master dataset holds ~2.6 million property records. A machine
   with at least 16GB RAM is recommended.

5. EXECUTION TIME: Step 09 (Bayesian target generation) now runs PyMC's NUTS
   sampler instead of brms/Stan — expect 10-30 minutes depending on hardware.

6. RANDOMNESS: R and NumPy random number generators produce different streams,
   so seeded draws (row sampling, rbinom, SMOTE, MCMC) are not bit-identical to
   the R run. The seeds are preserved so the Python pipeline is reproducible
   run-to-run, and the logic is identical.


---
### PIPELINE OVERVIEW (what each step does — Python edition)
---
```
  [REMOVED] Steps 01-06 — Stage 1 (raw data ingestion) and Stage 2 (master
    dataset construction). Their output, clean_master_combined.parquet, is the
    starting input for Step 07.

  Step 07 — EDA, outlier cleaning, derived variable construction
    Loads clean_master_combined.parquet, cleans hard errors and outliers,
    constructs derived variables (fuel_cost_ratio, co2_per_m2 etc.), computes
    correlation matrices. Produces: clean_master_final.parquet,
    mcmc_split_clean_master.parquet, ml_split_clean_master.parquet
    (R: dplyr/arrow/corrr → Python: pandas/pyarrow)

  Step 08 — Feature engineering for ML pipelines
    Information-gain feature ranking and stratified 80/20 train/test split.
    NOTE: operates on df_balanced from Step 10 — run 09 and 10 first (the
    original notebook had the same execution-order dependency).
    (R: FSelector/caret → Python: sklearn mutual_info_classif/train_test_split)

  Step 09 — Bayesian prior construction and 5-class risk label assignment
    Reads LILEE supplementary tables and mcmc_probabilities.xlsx, harmonises
    categories, combines severity-adjusted probabilities (geometric mean),
    draws a synthetic fuel_poor target, fits a Bayesian logistic regression,
    assigns 5-class predicted_class risk labels. Produces:
    ml_model_data_with_predictions.parquet, bayesian_fuel_poverty_model.nc
    (R: brms + cmdstanr/Stan → Python: PyMC + ArviZ)

  Step 10 — ML preprocessing: SMOTE, feature selection, train/test split
    Loads predictions parquet, renames columns, log transforms, caps outliers,
    dummy-encodes predictors, applies SMOTE balancing. Produces: df_balanced
    (R: fastDummies/recipes/themis → Python: pandas get_dummies / imblearn SMOTE)

  Step 11 — XGBoost (Pipeline A and B)          (R: xgboost → Python: xgboost)
  Step 12 — Random Forest (Pipeline A and B)    (R: randomForest → Python: sklearn)
  Step 13 — Deep Neural Network (Pipeline A and B)
                                                (R: keras/tensorflow → Python: tensorflow.keras)
  Step 14 — Multinomial Logistic Regression baseline (Pipeline A and B)
                                                (R: nnet::multinom → Python: sklearn LogisticRegression)

  Step 15 — Model evaluation
    Standardised evaluation across all models and pipelines: Accuracy,
    Macro F1, Sensitivity, Specificity, Kappa, multiclass ROC-AUC.
    (R: caret/MLmetrics/pROC → Python: sklearn.metrics)
```


---
### EXECUTION ORDER — Stages 3-7
---
*Stages 1-2 (Steps 01-06) were removed from this notebook: their outputs are
precomputed, and Stage 3 starts from `clean_master_combined.parquet`.*

**Run CELL 0 (environment setup) first**, then the steps in this order:

**STAGE 3: FEATURE ENGINEERING & EDA**
```
Step 07 — EDA, outlier cleaning, derived variable construction
```
**STAGE 4: TARGET VARIABLE GENERATION (BAYESIAN / MCMC)**
```
Step 09 — Bayesian prior construction and 5-class risk label assignment
```
**STAGE 5: ML PREPROCESSING**
```
Step 10 — SMOTE balancing and dummy encoding
Step 08 — Information gain feature selection and train/test split
          (depends on Step 10's df_balanced — same ordering as the original)
```
**STAGE 6: MODEL TRAINING**
```
Step 11 — XGBoost (Pipeline A and B)
Step 12 — Random Forest (Pipeline A and B)
Step 13 — Deep Neural Network (Pipeline A and B)   [requires tensorflow]
Step 14 — Multinomial Logistic Regression baseline (Pipeline A and B)
```
**STAGE 7: EVALUATION**
```
Step 15 — Model evaluation: Accuracy, F1, Kappa, ROC-AUC across all models
```


### CELL 0 — ENVIRONMENT SETUP

*    Change ```ROOT_PATH``` to point at your Database folder, then run this cell first.
*    Everything else in the notebook derives its paths from ```ROOT_PATH``` automatically.


In [ ]:
# --- 1. Mount Drive --- (Colab artifact removed — local run)
# --- 2. Set root path (ONLY LINE TO CHANGE) ---
import os
from pathlib import Path

ROOT_PATH = Path("/Users/hayder/Desktop/R Projects/ML_Project")

# --- 3. Derive all file paths from ROOT_PATH ---
def p(*parts) -> Path:
    return ROOT_PATH.joinpath(*parts)

# NOTE: Stage 1-2 raw-file paths (EPC CSVs, income, population, weather,
# energy-price files) were removed together with Steps 01-06. The pipeline
# now starts from the Stage 2 output (clean_master_combined.parquet).

# Fuel poverty prior and LILEE (used by Step 09)
LILEE_TABLES         = p("LILEE_supplementary_tables_2024.xlsx")
MCMC_PROBS           = p("mcmc_probabilities.xlsx")

# Intermediate pipeline outputs (clean_master_combined is the starting input;
# the rest are written during the run and read by later steps)
CLEAN_MASTER_PARQUET = p("clean_master_combined.parquet")
CLEAN_FINAL_CSV      = p("clean_master_final.csv")
CLEAN_FINAL_PARQUET  = p("clean_master_final.parquet")
ML_SPLIT_PARQUET     = p("ml_split_clean_master.parquet")
MCMC_SPLIT_PARQUET   = p("mcmc_split_clean_master.parquet")
ML_PREDICTIONS       = p("ml_model_data_with_predictions.parquet")
BAYESIAN_MODEL       = p("bayesian_fuel_poverty_model.nc")   # ArviZ InferenceData (replaces the .rds)
CORR_MATRIX          = p("correlation_matrix.csv")

# --- 4. Verify root path exists ---
if not ROOT_PATH.is_dir():
    raise FileNotFoundError(
        f"ROOT_PATH not found: {ROOT_PATH}\nCheck that the path above is correct."
    )

# Mirror the original R runner's setwd(): the step cells below read/write
# intermediate files with bare relative filenames, which land in ROOT_PATH.
os.chdir(ROOT_PATH)

print("Root path confirmed:", ROOT_PATH)
print("Files available:", len(list(ROOT_PATH.iterdir())))


### All packages already live inside each cell, this is a draft of all packages required, may not be run

In [ ]:
# Draft of all packages required — mirrors the original R package list.
# Install once (uncomment the line below), then restart the kernel:
# %pip install pandas numpy pyarrow openpyxl matplotlib scipy scikit-learn imbalanced-learn xgboost tensorflow pymc arviz

py_packages = [
    "pandas", "numpy", "pyarrow",   # data.table / dplyr / tidyr / arrow
    "openpyxl",                     # readxl
    "matplotlib", "scipy",          # ggplot2 / patchwork / gridExtra / corrplot
    "scikit-learn",                 # caret / FSelector / nnet / randomForest / MLmetrics / pROC
    "imbalanced-learn",             # smotefamily / themis / recipes (SMOTE)
    "xgboost",                      # xgboost / Matrix
    "tensorflow",                   # keras / tensorflow
    "pymc", "arviz",                # brms / cmdstanr / posterior / bayesplot
]
print("\n".join(py_packages))


## Step 07 — Exploratory Analysis, Outlier Cleaning, and Derived Variable Construction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

pd.set_option("display.max_columns", None)

# Stage 2 output — in the original notebook `clean_master` was in memory from
# Step 06; with Stages 1-2 cut, load it from the saved parquet instead.
clean_master = pd.read_parquet(CLEAN_MASTER_PARQUET)

print(clean_master.describe(include="all").T)   # summary(clean_master)

# ==========Clean for hard errors=========
# (R's filter() drops rows where a condition is NA; pandas comparisons with
# NaN evaluate False, so the same rows are dropped.)

clean_master = clean_master[
    # Positive values only
    (clean_master["income_bhc_2024"] > 0)
    & (clean_master["avg_fuel_bill_annual_individual_estimate"] > 0)
    & (clean_master["energy_consumption_current"] > 0)
    & (clean_master["CO2_EMISSIONS_CURRENT"] >= 0)

    # Valid EPC bands
    & (clean_master["epc_rating_numeric"] >= 1)
    & (clean_master["epc_rating_numeric"] <= 7)

    # Reasonable size and room count
    & (clean_master["total_floor_area"] > 10)
    & (clean_master["total_floor_area"] <= 500)
    & (clean_master["NUMBER_HABITABLE_ROOMS"] >= 1)
    & (clean_master["NUMBER_HABITABLE_ROOMS"] <= 20)

    # Valid insulation scale
    & (clean_master["insulation_score"] >= 1)
    & (clean_master["insulation_score"] <= 5)

    # No missing key values
    & clean_master["income_bhc_2024"].notna()
    & clean_master["avg_fuel_bill_annual_individual_estimate"].notna()
    & clean_master["epc_rating_numeric"].notna()
    & clean_master["insulation_score"].notna()
].copy()

# Create fuel cost ratio
clean_master["fuel_cost_ratio"] = (
    clean_master["avg_fuel_bill_annual_individual_estimate"] / clean_master["income_bhc_2024"]
)

# Drop invalid cost ratios
clean_master = clean_master[
    (clean_master["fuel_cost_ratio"] > 0) & (clean_master["fuel_cost_ratio"] <= 1)
].copy()

# Identify all numeric columns
numeric_cols = clean_master.select_dtypes(include="number").columns

# Compute quantiles (0% to 100% by 1%) for each numeric column
percentile_df = (
    clean_master[numeric_cols]
    .quantile(np.arange(0, 1.01, 0.01))
    .T.rename_axis("variable")
    .reset_index()
)

# Write to CSV
percentile_df.to_csv("percentile_summary_all_numeric_columns.csv", index=False)

# ==========Clean for outliers=========

# Treating anomalies in wall/roof efficiency scores
clean_master = clean_master[
    # Remove rows with invalid ordinal ratings
    (clean_master["ROOF_ENERGY_EFF"] >= 1) & (clean_master["ROOF_ENERGY_EFF"] <= 5)
    & (clean_master["WALLS_ENERGY_EFF"] >= 1) & (clean_master["WALLS_ENERGY_EFF"] <= 5)
    & (clean_master["insulation_score"] >= 1) & (clean_master["insulation_score"] <= 5)
].copy()

# Compute percentiles for capping
p99_energy = clean_master["ENERGY_CONSUMPTION_CURRENT"].quantile(0.99)
p99_fuel_bill = clean_master["avg_fuel_bill_annual_individual_estimate"].quantile(0.99)
p99_co2 = clean_master["CO2_EMISSIONS_CURRENT"].quantile(0.99)

# Apply fixes
# Remove invalid EPC efficiency
clean_master = clean_master[clean_master["CURRENT_ENERGY_EFFICIENCY"] <= 100].copy()

# Cap long-tail variables
clean_master["ENERGY_CONSUMPTION_CURRENT"] = clean_master["ENERGY_CONSUMPTION_CURRENT"].clip(upper=p99_energy)
clean_master["avg_fuel_bill_annual_individual_estimate"] = (
    clean_master["avg_fuel_bill_annual_individual_estimate"].clip(upper=p99_fuel_bill)
)
clean_master["CO2_EMISSIONS_CURRENT"] = clean_master["CO2_EMISSIONS_CURRENT"].clip(upper=p99_co2)

# Cap fuel bill difference to ±20,000
clean_master["fuel_bill_difference"] = clean_master["fuel_bill_difference"].clip(lower=-20000, upper=20000)

# Remove extreme edge case households
clean_master = clean_master[clean_master["fuel_cost_ratio"] <= 0.75].copy()

# ===========Categorical Variables =========

# Identify character or factor variables
cat_vars = clean_master.select_dtypes(include=["object", "category"]).columns.tolist()

# Display unique values for each
for var in cat_vars:
    print(f"\n Variable: {var}")
    print(clean_master[var].unique())

# Check for unique values and missing counts
for var in cat_vars:
    print(f"\n Variable: {var}")

    # Unique values
    print(clean_master[var].unique())

    col_lower = clean_master[var].astype("string").str.lower()

    # Count NAs
    print("🔸 NA values:", clean_master[var].isna().sum())

    # Count blanks and common placeholders
    print("🔸 Empty strings:", (clean_master[var] == "").sum())
    print("🔸 'unknown':", (col_lower == "unknown").sum())
    print("🔸 'no data!':", (col_lower == "no data!").sum())
    print("🔸 'invalid!':", (col_lower == "invalid!").sum())

# filter out unknown and invalid values
# (R's filter also drops NA TENURE / NA CONSTRUCTION_AGE_BAND rows — preserved)
tenure_lower = clean_master["TENURE"].astype("string").str.lower()
age_band_lower = clean_master["CONSTRUCTION_AGE_BAND"].astype("string").str.lower()
clean_master = clean_master[
    (tenure_lower != "unknown") & tenure_lower.notna()
    & (age_band_lower != "invalid!") & age_band_lower.notna()
].copy()

# derive features
clean_master["fuel_cost_ratio"] = (
    clean_master["avg_fuel_bill_annual_individual_estimate"] / clean_master["income_bhc_2024"]
)
clean_master["floor_area_per_room"] = clean_master["total_floor_area"] / clean_master["NUMBER_HABITABLE_ROOMS"]
clean_master["co2_per_m2"] = clean_master["CO2_EMISSIONS_CURRENT"] / clean_master["total_floor_area"]
clean_master["energy_per_m2"] = clean_master["energy_consumption_current"] / clean_master["total_floor_area"]
clean_master["energy_per_room"] = clean_master["energy_consumption_current"] / clean_master["NUMBER_HABITABLE_ROOMS"]

# =========== Clean parquet output =========

# Save updated clean_master
clean_master.to_parquet("clean_master_final.parquet")

# Save as plain CSV
clean_master.to_csv("clean_master_final.csv", index=False)

# Save as compressed CSV (gzip)
clean_master.to_csv("clean_master_final.csv.gz", index=False, compression="gzip")

# Save as Parquet (fast binary format)
clean_master.to_parquet("clean_master_final.parquet", compression="snappy")

print("Exported: CSV, Compressed CSV, and Parquet formats.")

# ==========EDA==========

# Get numeric variables
num_vars = clean_master.select_dtypes(include="number").columns.tolist()

# Loop through in batches of 4
for i in range(0, len(num_vars), 4):
    batch = num_vars[i : i + 4]

    fig, axes = plt.subplots(1, len(batch), figsize=(4 * len(batch), 3))
    axes = np.atleast_1d(axes)
    for ax, var in zip(axes, batch):
        vals = clean_master[var].dropna().to_numpy(dtype=float)
        try:
            kde = gaussian_kde(vals)
            xs = np.linspace(vals.min(), vals.max(), 200)
            ax.fill_between(xs, kde(xs), color="steelblue", alpha=0.5)
        except Exception:
            # zero-variance / degenerate columns: fall back to a histogram
            ax.hist(vals, bins=30, color="steelblue", alpha=0.5, density=True)
        ax.set_title(var, fontsize=10, fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
    plt.tight_layout()
    plt.show()

# =======correlation matrix for feature selection========

# 1. Select only numeric columns
numeric_vars = clean_master.select_dtypes(include="number")

# 2. Compute full correlation matrix (pairwise complete observations —
#    pandas' default, matching corrr::correlate)
cor_matrix = numeric_vars.corr().rename_axis("term").reset_index()

# 3. View or export the full correlation matrix
print(cor_matrix)

# Optional: Save as CSV
cor_matrix.to_csv("correlation_matrix_all_numeric.csv", index=False)

clean_master = pd.read_parquet("clean_master_final.parquet")

# ---- MCMC-based target generation variables ----


# MCMC split variables (LILEE-matching)
mcmc_vars = [
    "LMK_KEY", "local_authority_code.x", "epc_rating_numeric", "urban_rural", "region_name", "PROPERTY_TYPE",
    "CONSTRUCTION_AGE_BAND", "TOTAL_FLOOR_AREA", "fuel_category",
    "BUILT_FORM", "WALLS_ENERGY_EFF", "WALLS_ENERGY_EFF_LABEL",
    "WALLS_DESCRIPTION", "TENURE",
]

# ML modeling split variables (non-redundant, avoiding signal leakage)
ml_vars = [
    "LMK_KEY",
    "CURRENT_ENERGY_EFFICIENCY", "ENERGY_CONSUMPTION_CURRENT", "CO2_EMISSIONS_CURRENT",
    "BUILT_FORM", "local_authority_code.x", "LODGEMENT_DATE",
    "postcode_district", "insulation_score", "local_authority.x", "net_income_2024", "income_bhc_2024",
    "income_ahc_2024", "perc_low_income_households", "unemployment_rate",
    "imd_avg_2019", "hdd_01_20_median", "tas_winter_01_20_median",
    "tas_winter_1.5_median", "tas_winter_2_median", "tas_winter_2.5_median",
    "tas_winter_3.5_median", "tas_winter_4_median",
    "min_temp_01_20_median", "population_2024", "child_population",
    "senior_population", "working_age_population", "average_household_size_2023",
    "area_km", "population_density", "age_dependency_ratio", "total_floor_area",
    "electricity_avg_variable_per_kWh_price", "electricity_avg_fixed_cost_annual",
    "gas_avg_variable_per_kWh_price", "gas_avg_fixed_cost_annual",
    "avg_fuel_bill_annual_individual_estimate",
    "local_authority_mean_domessic_electricity_consumption_kWh_per_household",
    "local_authority_mean_domestic_gas_consumption_kWh_per_meter",
    "avg_fuel_bill_annual_local_average_estimate", "fuel_bill_difference",
    "fuel_cost_ratio", "floor_area_per_room", "co2_per_m2", "energy_per_m2",
    "energy_per_room",
]

# Filter columns
mcmc_data = clean_master[mcmc_vars].copy()
ml_data = clean_master[ml_vars].copy()

# Save to CSV
mcmc_data.to_csv("mcmc_split_clean_master.csv", index=False)
ml_data.to_csv("ml_split_clean_master.csv", index=False)

# Save as Parquet (optional and efficient)
mcmc_data.to_parquet("mcmc_split_clean_master.parquet")
ml_data.to_parquet("ml_split_clean_master.parquet")


## Step 08 — Feature Engineering for ML Pipelines

In [ ]:
# NOTE (execution order, preserved from the original notebook): this cell
# operates on `df_balanced`, which is produced by Step 10 (SMOTE balancing).
# Run Steps 09 and 10 first, then return here — the original R cell
# (`df_balanced <- df_balanced`) had exactly the same dependency.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

# Make sure target is a factor
df_balanced["predicted_class"] = df_balanced["predicted_class"].astype("category")

# Compute information gain
# (FSelector::information.gain ↔ mutual information between feature and class)
X_ig = df_balanced.drop(columns=["predicted_class"])
y_ig = df_balanced["predicted_class"].cat.codes
ig = pd.DataFrame(
    {"attr_importance": mutual_info_classif(X_ig, y_ig, random_state=42)},
    index=X_ig.columns,
)

# Sort descending
ig = ig.sort_values("attr_importance", ascending=False)

# Show top 20 features
print(ig.head(20))

# Optional: Plot top 15
ig["Feature"] = ig.index
top_ig = ig.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_ig["Feature"][::-1], top_ig["attr_importance"][::-1], color="darkgreen")
ax.set_title("Information Gain (Top 15 Features)")
ax.set_xlabel("Information Gain")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

# Ensure target is factor
df_balanced["predicted_class"] = df_balanced["predicted_class"].astype("category")

# Create stratified split: 80% train, 20% test
# (caret::createDataPartition(p = 0.8), set.seed(42))
train_data, test_data = train_test_split(
    df_balanced,
    test_size=0.2,
    stratify=df_balanced["predicted_class"],
    random_state=42,
)
train_data = train_data.copy()
test_data = test_data.copy()

# Check class balance in each
print(train_data["predicted_class"].value_counts(normalize=True))
print(test_data["predicted_class"].value_counts(normalize=True))

# Make sure target is a factor
train_data["predicted_class"] = train_data["predicted_class"].astype("category")
test_data["predicted_class"] = test_data["predicted_class"].astype("category")


## Step 09 — Bayesian Prior Construction and 5-Class Risk Label Assignment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

mcmc_data = pd.read_parquet("mcmc_split_clean_master.parquet")

# File path
supp_file = "LILEE_supplementary_tables_2024.xlsx"

# Only required sheets
selected_sheets = [
    "Table_2_FPEER", "Table_3_SAP", "Table_4_rurality", "Table_5_region", "Table_6_dwelling_type",
    "Table_7_age_dwelling", "Table_8_floor_area", "Table_10_main_fuel",
    "Table_11_wall_insulation", "Table_12_tenure",
]

# Load selected sheets (the R version list2env'd them into the global
# environment as reference tables; kept here as a dict)
lilee_sheets = {s: pd.read_excel(supp_file, sheet_name=s) for s in selected_sheets}

# --- urban_rural simplification ---
mcmc_data["urban_rural_simplified"] = np.select(
    [
        mcmc_data["urban_rural"].isin([
            "Urban: Majority nearer to a major town or city",
            "Urban: Majority further from a major town or city",
        ]),
        mcmc_data["urban_rural"].isin([
            "Intermediate urban: Majority nearer to a major town or city",
            "Intermediate urban: Majority further from a major town or city",
            "Intermediate rural: Majority nearer to a major town or city",
            "Intermediate rural: Majority further from a major town or city",
        ]),
        mcmc_data["urban_rural"].isin([
            "Majority rural: Majority nearer to a major town or city",
            "Majority rural: Majority further from a major town or city",
        ]),
    ],
    ["Urban", "Semi-rural", "Rural"],
    default=None,  # catch any unexpected values
)

# Convert epc_rating_numeric to EPC band letters
mcmc_data["epc_rating_band"] = mcmc_data["epc_rating_numeric"].map(
    {1: "A", 2: "B", 3: "C", 4: "D", 5: "E", 6: "F", 7: "G"}
)

# --- dwelling type mapping (LILEE) ---
pt = mcmc_data["PROPERTY_TYPE"]
bf = mcmc_data["BUILT_FORM"]
terrace_forms = ["Mid-Terrace", "End-Terrace", "Enclosed Mid-Terrace", "Enclosed End-Terrace"]
end_terrace_forms = ["End-Terrace", "Enclosed End-Terrace"]
mid_terrace_forms = ["Mid-Terrace", "Enclosed Mid-Terrace"]

mcmc_data["dwelling_type_mapped"] = np.select(
    [
        # Maisonette — always Converted flat per LILEE Note A
        pt == "Maisonette",
        # Flat in a terrace = subdivision of an original terraced house
        (pt == "Flat") & bf.isin(terrace_forms),
        # Flat in semi-detached or detached building = purpose-built block
        (pt == "Flat") & bf.isin(["Semi-Detached", "Detached"]),
        # Flat with blank/NA BUILT_FORM
        (pt == "Flat") & (bf.isna() | (bf.astype("string").str.strip() == "").fillna(False)),
        # House — BUILT_FORM determines dwelling type directly
        (pt == "House") & (bf == "Detached"),
        (pt == "House") & (bf == "Semi-Detached"),
        (pt == "House") & bf.isin(end_terrace_forms),
        (pt == "House") & bf.isin(mid_terrace_forms),
        # Bungalow — same BUILT_FORM logic as House
        (pt == "Bungalow") & (bf == "Detached"),
        (pt == "Bungalow") & (bf == "Semi-Detached"),
        (pt == "Bungalow") & bf.isin(end_terrace_forms),
        (pt == "Bungalow") & bf.isin(mid_terrace_forms),
        # Park home — not in LILEE dwelling stock
        pt == "Park home",
    ],
    [
        "Converted flat",
        "Converted flat",
        "Purpose-built flat",
        None,
        "Detached",
        "Semi-detached",
        "End terrace",
        "Mid terrace",
        "Detached",
        "Semi-detached",
        "End terrace",
        "Mid terrace",
        None,
    ],
    default=None,
)
mcmc_data = mcmc_data[mcmc_data["dwelling_type_mapped"].notna()].copy()
print("dwelling_type_mapped distribution:")
print(mcmc_data["dwelling_type_mapped"].value_counts())
print("Rows remaining after filter:", len(mcmc_data))

# --- dwelling age mapping ---
age = mcmc_data["CONSTRUCTION_AGE_BAND"]
mcmc_data["dwelling_age_mapped"] = np.select(
    [
        age.isin(["England and Wales: before 1900", "England and Wales: 1900-1929"]),
        age == "England and Wales: 1930-1949",
        age == "England and Wales: 1950-1966",
        age.isin(["England and Wales: 1967-1975", "England and Wales: 1976-1982"]),
        age == "England and Wales: 1983-1990",
        age.isin(["England and Wales: 1991-1995", "England and Wales: 1996-2002"]),
        age.isin(["England and Wales: 2003-2006", "England and Wales: 2007 onwards",
                  "England and Wales: 2007-2011", "England and Wales: 2012 onwards"]),
    ],
    ["Pre 1919", "1919 to 1944", "1945 to 1964", "1965 to 1980",
     "1981 to 1990", "1991 to 2002", "Post 2002"],
    default=None,
)

# Remove rows that couldn't be mapped
mcmc_data = mcmc_data[mcmc_data["dwelling_age_mapped"].notna()].copy()

# --- wall insulation category (regex on WALLS_DESCRIPTION, case-insensitive) ---
wd = mcmc_data["WALLS_DESCRIPTION"]
def _wd(pat):
    return wd.str.contains(pat, case=False, regex=True, na=False)

mcmc_data["wall_insulation_cat"] = np.select(
    [
        # Cavity with insulation
        _wd(r"cavity wall, filled cavity") & ~_wd(r"partial"),
        _wd(r"cavity wall.*(?:insulated|with.*insulation)") & ~_wd(r"no insulation|partial"),
        _wd(r"cavity wall, filled cavity and"),
        # Cavity uninsulated
        _wd(r"cavity wall.*no insulation"),
        _wd(r"cavity wall.*partial insulation"),
        # Solid with insulation
        _wd(r"solid brick.*(?:insulated|with.*insulation)") & ~_wd(r"no insulation"),
        # Solid uninsulated
        _wd(r"solid brick.*no insulation"),
        _wd(r"solid brick.*partial insulation"),
        wd.str.strip().str.fullmatch(r"solid brick, as built", case=False).fillna(False),
        # Other — stone, timber, system built, cob, park home, U-value declarations
        _wd(r"sandstone|limestone|granite|whinstone"),
        _wd(r"timber frame"),
        _wd(r"system built"),
        _wd(r"cob"),
        _wd(r"park home"),
        _wd(r"average thermal transmittance|trawsyriannedd thermol"),
    ],
    ["cavity with insulation", "cavity with insulation", "cavity with insulation",
     "cavity uninsulated", "cavity uninsulated",
     "solid with insulation",
     "solid uninsulated", "solid uninsulated", "solid uninsulated",
     "other", "other", "other", "other", "other", "other"],
    default="other",
)

# --- tenure category ---
mcmc_data["tenure_cat"] = np.select(
    [
        mcmc_data["TENURE"].isin(["owner-occupied", "Owner-occupied"]),
        mcmc_data["TENURE"].isin(["rental (private)", "Rented (private)"]),
        mcmc_data["TENURE"].isin(["rental (social)", "Rented (social)"]),
    ],
    ["Owner occupied", "Private rented", "Social housing"],
    default=None,
)
mcmc_data = mcmc_data[mcmc_data["tenure_cat"].notna()].copy()  # remove rows where tenure was not defined

# Categorize total_floor_area before joining
tfa = mcmc_data["TOTAL_FLOOR_AREA"]
mcmc_data["total_floor_area_bin"] = np.select(
    [
        tfa < 50,
        (tfa >= 50) & (tfa < 70),
        (tfa >= 70) & (tfa < 90),
        (tfa >= 90) & (tfa < 110),
        tfa >= 110,
    ],
    ["Less than 50 sqm", "50 to 69 sqm", "70 to 89 sqm", "90 to 109 sqm", "110 sqm or more"],
    default=None,
)

# --- 2. Sample a small chunk from your harmonized data ---
model_data = mcmc_data.sample(n=10000, random_state=123)  # set.seed(123)

# --- 3. Load probability table (one sheet with all categories + severity-adjusted probs) ---
prob_table = pd.read_excel(MCMC_PROBS)
prob_table["feature"] = prob_table["feature"].astype("string").str.strip().str.lower()
prob_table["feature_value"] = prob_table["feature_value"].astype("string").str.strip().str.lower()

# --- 4. Match on keys and get severity-adjusted probabilities ---

# Check if column names are harmonized
prob_table.columns = [c.lower().replace(" ", "_") for c in prob_table.columns]

# Merge with MCMC features one by one (by multiple keys)
keys = [
    "epc_rating_band", "urban_rural_simplified", "region_name",
    "dwelling_type_mapped", "dwelling_age_mapped", "total_floor_area_bin",
    "fuel_category", "wall_insulation_cat", "tenure_cat",
]

# Convert keys in model_data to lower-case for join compatibility
# ("string" dtype preserves missing values through .str.lower(), like tolower(NA) in R)
model_data_join = model_data.copy()
for k in keys:
    model_data_join[k] = model_data_join[k].astype("string").str.lower()

# Fix region name: LA lookup uses "east of england", LILEE uses "east"
model_data_join["region_name"] = model_data_join["region_name"].where(
    model_data_join["region_name"] != "east of england", "east"
)

print("Region name values after fix:")
print(model_data_join["region_name"].value_counts())

# Join using keys (prob_table has a "feature" and "feature_value" column)
combined = model_data_join

for feature in keys:
    temp_prob = prob_table.loc[
        prob_table["feature"] == feature, ["feature_value", "severity_adjusted_probability"]
    ].copy()
    temp_prob.columns = [feature, f"{feature}_prob"]

    # Merge this feature's probability into the dataset
    combined = combined.merge(temp_prob, on=feature, how="left")

# --- 5. Combine the individual probabilities (geometric mean) ---
prob_cols = [c for c in combined.columns if c.endswith("_prob")]
probs_mat = combined[prob_cols].to_numpy(dtype=float)
has_na = np.isnan(probs_mat).any(axis=1)
x_floored = np.maximum(probs_mat, 0.001)  # Floor at 0.001: safety net for EPC bands A/B/C
combined["combined_prob"] = np.where(has_na, np.nan, np.exp(np.log(x_floored).mean(axis=1)))

print("combined_prob distribution after floor:")
print(combined["combined_prob"].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.9, 0.95, 1]))
print("Zero combined_prob values:", int((combined["combined_prob"] == 0).sum()))
print("NA combined_prob values:", int(combined["combined_prob"].isna().sum()))

# --- 6. Generate synthetic target from LILEE prior (combined_prob) ---
# No likelihood update applied. WFP rejected (SHAP rank 38/48, gain=0.003).
# DESNZ sub-regional statistics reserved for external validation.
rng = np.random.default_rng(123)
cp = combined["combined_prob"].to_numpy(dtype=float)
fuel_poor = np.full(len(cp), np.nan)
ok = ~np.isnan(cp)
fuel_poor[ok] = rng.binomial(1, cp[ok])  # rbinom(n, 1, combined_prob)
combined["fuel_poor"] = fuel_poor

model_data = combined[combined["fuel_poor"].notna() & combined["combined_prob"].notna()].copy()
model_data["fuel_poor"] = model_data["fuel_poor"].astype(int)

print(f"\nRows retained for model: {len(model_data)}")
print("fuel_poor distribution:")
print(model_data["fuel_poor"].value_counts().sort_index())

# --- 7. Ensure all categorical vars are factors ---
cat_vars = keys
for c in cat_vars:
    model_data[c] = model_data[c].astype("category")

# --- 8. Fit Bayesian model using PyMC (replaces {brms} + cmdstanr backend) ---
import pymc as pm
import arviz as az

# Treatment-coded design matrix: first (alphabetical) level of each factor is
# the reference level — the same coding R's model formula produces.
X_design = pd.get_dummies(model_data[cat_vars], drop_first=True, dtype=float)
coef_names = X_design.columns.tolist()
X_mat = X_design.to_numpy()
y_obs = model_data["fuel_poor"].to_numpy()

# formula: fuel_poor ~ epc_rating_band + urban_rural_simplified + region_name +
#          dwelling_type_mapped + dwelling_age_mapped + total_floor_area_bin +
#          fuel_category + wall_insulation_cat + tenure_cat,  family = bernoulli()
#
# Regularising priors prevent separation on EPC bands.
# Normal(0,1) on log-odds scale keeps coefficients in a plausible range.
# The intercept gets a wider prior since fuel poverty base rate is genuinely low.
# (brms applies the Intercept prior to the centred-predictor intercept; here it
# is on the uncentred intercept — a reparameterisation of the same model.)
with pm.Model(coords={"coef": coef_names}) as bayes_model:
    intercept = pm.Normal("Intercept", mu=-3, sigma=2)   # prior(normal(-3, 2), class = Intercept)
    b = pm.Normal("b", mu=0, sigma=1, dims="coef")       # prior(normal(0, 1), class = b)
    eta = intercept + pm.math.dot(X_mat, b)
    pm.Bernoulli("fuel_poor", p=pm.math.sigmoid(eta), observed=y_obs)

    # chains = 4, iter = 4000, warmup = 2000 → 2000 post-warmup draws per chain
    fit_bayes = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,             # 4 chains to catch non-mixing
        cores=4,
        random_seed=42,
        target_accept=0.95,   # adapt_delta = 0.95: reduces divergences
        max_treedepth=15,     # fixes the treedepth warning
    )

# --- 9. Inspect ---
print(az.summary(fit_bayes))          # summary(fit_bayes)
az.plot_trace(fit_bayes)              # plot(fit_bayes)
plt.tight_layout()
plt.show()

with bayes_model:
    ppc = pm.sample_posterior_predictive(fit_bayes, random_seed=42)
az.plot_ppc(ppc)                      # pp_check(fit_bayes)
plt.show()

# bayesplot::mcmc_rhat(rhat(fit_bayes))
rhats = az.rhat(fit_bayes)
rhat_vals = np.concatenate([np.atleast_1d(rhats[v].values).ravel() for v in rhats.data_vars])
plt.figure(figsize=(6, 4))
plt.barh(range(len(rhat_vals)), np.sort(rhat_vals), color="steelblue")
plt.axvline(1.0, linestyle="--", color="grey")
plt.axvline(1.05, linestyle=":", color="red")
plt.xlabel("R-hat")
plt.title("mcmc_rhat")
plt.show()

# --- 10. Save the model ---
fit_bayes.to_netcdf("bayesian_fuel_poverty_model.nc")   # saveRDS(fit_bayes, ".rds")

# --- Step 1: Get predicted probabilities from fitted model ---
# posterior_epred(fit_bayes): posterior draws × observations, computed
# chain-by-chain to keep memory bounded.
# --- Step 2: Take the mean predicted probability per observation ---
post = fit_bayes.posterior
n_chains = post.sizes["chain"]
mean_probs = np.zeros(len(model_data))
for ch in range(n_chains):
    b_ch = post["b"].isel(chain=ch).to_numpy()           # (draws, n_coefs)
    a_ch = post["Intercept"].isel(chain=ch).to_numpy()   # (draws,)
    mean_probs += expit(X_mat @ b_ch.T + a_ch).mean(axis=1)
mean_probs /= n_chains

plt.figure()
plt.hist(mean_probs, bins=30, color="skyblue", edgecolor="black")
plt.axvline(np.median(mean_probs), linestyle="--", color="red")
plt.title("Distribution of Mean Predicted Probabilities")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Frequency")
plt.show()

print(pd.Series(mean_probs).quantile([0.25, 0.5, 0.75, 0.9, 0.95]))

# Set thresholds using your quantile output
thresholds = [-np.inf, 0.00004, 0.0376, 0.0619, 0.0791, np.inf]

# Assign ordinal risk categories
model_data["predicted_class"] = pd.cut(
    mean_probs,
    bins=thresholds,
    labels=["Very Low", "Low", "Medium", "High", "Very High"],
    right=True,
    include_lowest=True,
)

# View class distribution
print(model_data["predicted_class"].value_counts().sort_index())

# Ensure LMK_KEY is character for both datasets
model_data["LMK_KEY"] = model_data["LMK_KEY"].astype(str)
ml_data["LMK_KEY"] = ml_data["LMK_KEY"].astype(str)

# Join only predicted_class from model_data
ml_model_data = ml_data.merge(
    model_data[["LMK_KEY", "predicted_class"]], on="LMK_KEY", how="left"
)

# Optional: check distribution of classes
print(ml_model_data["predicted_class"].value_counts(dropna=False))

ml_model_data = ml_model_data.dropna()   # filter(complete.cases(.))

# Save the final dataset with predictions
ml_model_data.to_parquet("ml_model_data_with_predictions.parquet")
ml_model_data.to_csv("ml_model_data_with_predictions.csv", index=False)


## Step 10 — ML Preprocessing: SMOTE, Feature Selection, and Train/Test Split

In [ ]:
import math
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load your data
df = pd.read_parquet("ml_model_data_with_predictions.parquet")

# Rename selected columns to lowercase
df = df.rename(columns={
    "CURRENT_ENERGY_EFFICIENCY": "current_energy_efficiency",
    "ENERGY_CONSUMPTION_CURRENT": "energy_consumption_current",
    "CO2_EMISSIONS_CURRENT": "co2_emissions_current",
    "BUILT_FORM": "built_form",
})

# (pandas rename skips keys that aren't present; the WFP column below only
# exists in older versions of the predictions file)
df = df.rename(columns={
    "electricity_avg_variable_per_kWh_price": "elec_price_kwh",
    "num_households_received_winter_fuel_payment_2023": "hh_winter_fuel_payment_2023",
    "avg_fuel_bill_annual_individual_estimate": "fuel_bill_individual_est",
    "local_authority_mean_domessic_electricity_consumption_kWh_per_household": "la_elec_use_kwh_hh",
    "local_authority_mean_domestic_gas_consumption_kWh_per_meter": "la_gas_use_kwh_meter",
    "avg_fuel_bill_annual_local_average_estimate": "fuel_bill_local_avg_est",
})

# Convert to factor
df["predicted_class"] = df["predicted_class"].astype("category")

# Class distribution
print(df["predicted_class"].value_counts())
print(df["predicted_class"].value_counts(normalize=True))

# Bar plot
df["predicted_class"].value_counts().plot(kind="bar", color="steelblue")
plt.title("Distribution of Target Variable")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Check missing values
missing_summary = df.isna().sum()
print(missing_summary[missing_summary > 0])

# Select numeric columns
df_numeric = df.select_dtypes(include="number")

# Summary statistics for numeric columns
summary_stats = df_numeric.describe()
# Print summary statistics
print(summary_stats)

# =======Remedies based on summary stats=======
# log-transform energy_per_room
df["log_energy_per_room"] = np.log(df["energy_per_room"] + 1)

# Remove `fuel_bill_difference`
df = df.drop(columns=["fuel_bill_difference"])
df = df.drop(columns=["local_authority.x"])

# Cap `floor_area_per_room` to 99th percentile and remove outliers
floor_cap = df["floor_area_per_room"].quantile(0.99)
df = df[df["floor_area_per_room"] <= floor_cap].copy()

# Cap `fuel_cost_ratio` to 99th percentile and remove extreme values
fuel_ratio_cap = df["fuel_cost_ratio"].quantile(0.99)
df = df[df["fuel_cost_ratio"] <= fuel_ratio_cap].copy()

# Log-transform `fuel_bill_individual_est`
df["log_fuel_bill_individual_est"] = np.log(df["fuel_bill_individual_est"] + 1)  # +1 to avoid log(0)

# List of population-related columns
population_vars = [
    "population_2024",
    "child_population",
    "senior_population",
    "working_age_population",
]

# Loop to filter rows beyond the 95th percentile for any of these columns
# (sequential, as in the original — each cap is computed on the already-filtered frame)
for var in population_vars:
    cap = df[var].quantile(0.95)
    df = df[df[var] <= cap].copy()

# Drop extra TAS winter scenario columns (keep only 1.5 and 3.5)
tas_cols_to_keep = ["tas_winter_1.5_median", "tas_winter_3.5_median"]
tas_cols_to_drop = [c for c in df.columns if re.match(r"^tas_winter_.*_median$", c)]
tas_cols_to_drop = [c for c in tas_cols_to_drop if c not in tas_cols_to_keep]
df = df.drop(columns=tas_cols_to_drop)

# ========= EDA: Histograms and Correlation Matrix ==========

# Ensure output folder exists
os.makedirs("eda_histograms", exist_ok=True)

# Select numeric columns
var_names = df.select_dtypes(include="number").columns.tolist()

# Batch settings
batch_size = 9
num_batches = math.ceil(len(var_names) / batch_size)

# Loop through batches
for i in range(1, num_batches + 1):
    print(f"Saving histogram batch {i} of {num_batches}")

    start = (i - 1) * batch_size
    vars_subset = var_names[start : start + batch_size]

    fig, axes = plt.subplots(3, 3, figsize=(8, 8), dpi=200)
    for ax, v in zip(axes.ravel(), vars_subset):
        ax.hist(df[v].dropna(), bins=30, color="steelblue", edgecolor="white")
        ax.set_xlabel(v)
        ax.set_ylabel("Count")
    for ax in axes.ravel()[len(vars_subset):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(f"eda_histograms/hist_batch_{i}.png")
    plt.close(fig)

# Correlation matrix
df_numeric = df.select_dtypes(include="number")
cor_matrix = df_numeric.corr()   # pairwise complete observations
cor_matrix.to_csv("correlation_matrix.csv")

# Plot correlation matrix
# Define 15 selected variables
vars_selected = [
    "current_energy_efficiency",
    "fuel_cost_ratio",
    "fuel_bill_individual_est",
    "income_ahc_2024",
    "imd_avg_2019",
    "perc_low_income_households",
    "tas_winter_3.5_median",
    "elec_price_kwh",
    "insulation_score",
    "hdd_01_20_median",
    "senior_population",
    "average_household_size_2023",
    "total_floor_area",
    "floor_area_per_room",
    "gas_avg_variable_per_kWh_price",
]

# Subset and compute correlation
cor_subset = df[vars_selected].corr()

# corrplot(method="color", type="upper", order="hclust") equivalent
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from matplotlib.colors import LinearSegmentedColormap

dist_mat = 1 - cor_subset.to_numpy()
np.fill_diagonal(dist_mat, 0.0)
order = leaves_list(linkage(squareform(dist_mat, checks=False), method="complete"))
cor_ord = cor_subset.iloc[order, order]

cmap = LinearSegmentedColormap.from_list("mwn", ["maroon", "white", "navy"], N=200)  # your palette
mask = np.tril(np.ones_like(cor_ord, dtype=bool), k=-1)   # type = "upper"
plot_mat = np.ma.masked_array(cor_ord.to_numpy(), mask=mask)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(plot_mat, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(cor_ord)))
ax.set_xticklabels(cor_ord.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(cor_ord)))
ax.set_yticklabels(cor_ord.index, fontsize=7)
ax.set_xticks(np.arange(-0.5, len(cor_ord)), minor=True)
ax.set_yticks(np.arange(-0.5, len(cor_ord)), minor=True)
ax.grid(which="minor", color="black", linewidth=0.3)   # thinner and softer grid
fig.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.show()

# ========= Real preprocessing begins=========
df = df.drop(columns=["postcode_district", "local_authority_code.x", "LMK_KEY"])
df = df.drop(columns=["LODGEMENT_DATE"])

# Convert categorical variables to factors
# Identify character columns
char_vars = df.select_dtypes(include="object").columns.tolist()

# Convert character columns to factor in a loop
for c in char_vars:
    df[c] = df[c].astype("category")

# Ensure target is a factor
df["predicted_class"] = df["predicted_class"].astype("category")

# Step 1: Separate target before encoding
target = df["predicted_class"]
df_nontarget = df.drop(columns=["predicted_class"])

# Step 2: Dummy encode only predictors
# (fastDummies::dummy_cols(remove_first_dummy = TRUE) ↔ get_dummies(drop_first=True))
df_encoded = pd.get_dummies(df_nontarget, drop_first=True, dtype=int)

# Step 3: Reattach the target column
df_ready = pd.concat([df_encoded, target.rename("predicted_class")], axis=1)

# SMOTE balancing (recipes + themis::step_smote ↔ imblearn SMOTE: every class
# is upsampled to the majority class size, k = 5 neighbours in both)
from imblearn.over_sampling import SMOTE

X_smote = df_ready.drop(columns=["predicted_class"])
y_smote = df_ready["predicted_class"]
smote = SMOTE(random_state=42)  # the R run had no explicit seed; fixed here for reproducibility
X_bal, y_bal = smote.fit_resample(X_smote, y_smote)
df_balanced = X_bal.copy()
df_balanced["predicted_class"] = y_bal

# Check distribution of the target variable
print(df_balanced["predicted_class"].value_counts())

# View as proportions (optional)
print(df_balanced["predicted_class"].value_counts(normalize=True))


## Step 11 — XGBoost Classifier (Pipeline A and B)

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Convert class to numeric labels for XGBoost (0-indexed)
# (category order comes from the parquet dictionary — the R cut() level
# order Very Low..Very High — preserved identically by R factor() and pandas)
label_map = list(train_data["predicted_class"].cat.categories)
train_label = train_data["predicted_class"].cat.codes.to_numpy()
test_label = test_data["predicted_class"].cat.codes.to_numpy()

# Define feature sets
pipeline_a_vars = [c for c in train_data.columns if c != "predicted_class"]  # includes efficiency
pipeline_b_vars = [c for c in pipeline_a_vars if c != "current_energy_efficiency"]

# Add high-correlation vars back into B
pipeline_b_vars = list(dict.fromkeys(
    pipeline_b_vars + ["energy_consumption_current", "co2_per_m2", "energy_per_m2"]
))

# Ensure no missing
train_a = train_data[pipeline_a_vars + ["predicted_class"]]
train_b = train_data[pipeline_b_vars + ["predicted_class"]]

test_a = test_data[pipeline_a_vars + ["predicted_class"]]
test_b = test_data[pipeline_b_vars + ["predicted_class"]]

# Create DMatrix for Pipeline A
dtrain_a = xgb.DMatrix(train_a.drop(columns=["predicted_class"]).to_numpy(dtype=float), label=train_label)
dtest_a = xgb.DMatrix(test_a.drop(columns=["predicted_class"]).to_numpy(dtype=float), label=test_label)

# Create DMatrix for Pipeline B
dtrain_b = xgb.DMatrix(train_b.drop(columns=["predicted_class"]).to_numpy(dtype=float), label=train_label)
dtest_b = xgb.DMatrix(test_b.drop(columns=["predicted_class"]).to_numpy(dtype=float), label=test_label)

# ======== Train XGBoost Model for Pipeline A ========
params = {
    "objective": "multi:softprob",
    "num_class": len(label_map),
    "eval_metric": "mlogloss",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42,   # set.seed(42)
}

xgb_model_a = xgb.train(
    params=params,
    dtrain=dtrain_a,
    num_boost_round=150,
    evals=[(dtrain_a, "train"), (dtest_a, "test")],   # watchlist
    early_stopping_rounds=10,
    verbose_eval=1,
)

# ======== Train XGBoost Model for Pipeline B ========
xgb_model_b = xgb.train(
    params=params,
    dtrain=dtrain_b,
    num_boost_round=150,
    evals=[(dtrain_b, "train"), (dtest_b, "test")],
    early_stopping_rounds=10,
    verbose_eval=1,
)

# ======== Evaluate Pipeline A on Test Set ========

# Predict probabilities (the Python API already returns an (n, num_class) matrix,
# so no reshape is needed)
pred_prob_matrix_a = xgb_model_a.predict(dtest_a)

# Get predicted class index
pred_class_index_a = pred_prob_matrix_a.argmax(axis=1)

# Convert to categorical with original class labels
pred_class_a = pd.Categorical([label_map[i] for i in pred_class_index_a], categories=label_map)

# True labels
true_class = test_data["predicted_class"]

# Confusion matrix (caret orientation: rows = Prediction, cols = Reference)
cm_a = (
    pd.crosstab(
        pd.Series(pred_class_a, name="Prediction"),
        pd.Series(true_class.to_numpy(), name="Reference"),
    )
    .reindex(index=label_map, columns=label_map, fill_value=0)
)
print(cm_a)
print(classification_report(true_class, pred_class_a, zero_division=0))

# Accuracy
accuracy_a = accuracy_score(true_class, pred_class_a)
print(f"\nAccuracy: {round(accuracy_a, 4)}")

# Mirror of the Pipeline A prediction block for Pipeline B — `pred_class_b`
# is consumed by Step 15 (the original notebook references it there)
pred_prob_matrix_b = xgb_model_b.predict(dtest_b)
pred_class_index_b = pred_prob_matrix_b.argmax(axis=1)
pred_class_b = pd.Categorical([label_map[i] for i in pred_class_index_b], categories=label_map)


## Step 12 — Random Forest Classifier (Pipeline A and B)

In [ ]:
# ===== Load Required Library =====
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ===== Prepare Training & Test Data =====

# Pipeline A
rf_train_a = train_data[pipeline_a_vars + ["predicted_class"]]
rf_test_a = test_data[pipeline_a_vars + ["predicted_class"]]

# Pipeline B
rf_train_b = train_data[pipeline_b_vars + ["predicted_class"]]
rf_test_b = test_data[pipeline_b_vars + ["predicted_class"]]

# ===== Train Random Forest - Pipeline A =====
# randomForest(ntree = 300); mtry default floor(sqrt(p)) ↔ max_features="sqrt"
rf_model_a = RandomForestClassifier(n_estimators=300, random_state=42)  # set.seed(42)
rf_model_a.fit(rf_train_a[pipeline_a_vars], rf_train_a["predicted_class"])

# Predict on test set A
rf_pred_a = pd.Categorical(rf_model_a.predict(rf_test_a[pipeline_a_vars]), categories=label_map)

# Accuracy for Pipeline A
rf_acc_a = float((np.asarray(rf_pred_a) == rf_test_a["predicted_class"].to_numpy()).mean())
print("Random Forest - Pipeline A Accuracy:", round(rf_acc_a, 4))


# ===== Train Random Forest - Pipeline B =====
rf_model_b = RandomForestClassifier(n_estimators=300, random_state=42)  # set.seed(42)
rf_model_b.fit(rf_train_b[pipeline_b_vars], rf_train_b["predicted_class"])

# Predict on test set B
rf_pred_b = pd.Categorical(rf_model_b.predict(rf_test_b[pipeline_b_vars]), categories=label_map)

# Accuracy for Pipeline B
rf_acc_b = float((np.asarray(rf_pred_b) == rf_test_b["predicted_class"].to_numpy()).mean())
print("Random Forest - Pipeline B Accuracy:", round(rf_acc_b, 4))

# ==========================
# 📦 Pipeline A Evaluation
# ==========================

# Confusion Matrix (caret orientation: rows = Prediction, cols = Reference)
cm_rf_a = (
    pd.crosstab(
        pd.Series(rf_pred_a, name="Prediction"),
        pd.Series(rf_test_a["predicted_class"].to_numpy(), name="Reference"),
    )
    .reindex(index=label_map, columns=label_map, fill_value=0)
)
print(cm_rf_a)
print(classification_report(rf_test_a["predicted_class"], rf_pred_a, zero_division=0))

# Macro F1
# (The original's fp/fn are transposed relative to the usual convention, which
# swaps precision and recall per class — F1 is invariant to that swap, so the
# loop is preserved verbatim.)
label_map = list(train_data["predicted_class"].cat.categories)
f1_per_class_a = []

for cls in label_map:
    tp = cm_rf_a.loc[cls, cls]
    fp = cm_rf_a[cls].sum() - tp        # sum(cm$table[, cls]) - tp
    fn = cm_rf_a.loc[cls].sum() - tp    # sum(cm$table[cls, ]) - tp
    precision = 0 if tp + fp == 0 else tp / (tp + fp)
    recall = 0 if tp + fn == 0 else tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    f1_per_class_a.append(f1)

macro_f1_rf_a = float(np.mean(f1_per_class_a))
print("Random Forest - Pipeline A Macro F1:", round(macro_f1_rf_a, 4))


# ==========================
# 📦 Pipeline B Evaluation
# ==========================

# Confusion Matrix
cm_rf_b = (
    pd.crosstab(
        pd.Series(rf_pred_b, name="Prediction"),
        pd.Series(rf_test_b["predicted_class"].to_numpy(), name="Reference"),
    )
    .reindex(index=label_map, columns=label_map, fill_value=0)
)
print(cm_rf_b)
print(classification_report(rf_test_b["predicted_class"], rf_pred_b, zero_division=0))

# Macro F1
f1_per_class_b = []

for cls in label_map:
    tp = cm_rf_b.loc[cls, cls]
    fp = cm_rf_b[cls].sum() - tp
    fn = cm_rf_b.loc[cls].sum() - tp
    precision = 0 if tp + fp == 0 else tp / (tp + fp)
    recall = 0 if tp + fn == 0 else tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    f1_per_class_b.append(f1)

macro_f1_rf_b = float(np.mean(f1_per_class_b))
print("Random Forest - Pipeline B Macro F1:", round(macro_f1_rf_b, 4))


## Step 13 — Deep Neural Network (Pipeline A and B)

In [ ]:
# ======================
# 1. Load Libraries
# ======================
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# (The original required the "batnet" conda env via reticulate; a plain
# tensorflow install suffices in Python.)

# ======================
# 2. Prepare Data
# ======================

# Assume train_data and test_data are already available
# And target variable is `predicted_class`

# Convert factor to numeric labels (0-based)
train_label = train_data["predicted_class"].cat.codes.to_numpy()
test_label = test_data["predicted_class"].cat.codes.to_numpy()

# One-hot encode labels
y_train_cat = keras.utils.to_categorical(train_label)
y_test_cat = keras.utils.to_categorical(test_label)
num_classes = y_train_cat.shape[1]

# Select top 20 features (based on earlier SHAP/info gain results)
top_features = [
    "current_energy_efficiency", "insulation_score", "co2_emissions_current",
    "fuel_bill_individual_est", "fuel_cost_ratio", "fuel_bill_local_avg_est",
    "elec_price_kwh", "gas_avg_fixed_cost_annual", "energy_per_room",
    "electricity_avg_fixed_cost_annual", "log_energy_per_room", "income_bhc_2024",
    "area_km", "energy_per_m2", "population_density", "senior_population",
    "built_form_Mid-Terrace", "built_form_Semi-Detached",
]

# Filter features
X_train_a = train_data[top_features].to_numpy(dtype=float)
X_test_a = test_data[top_features].to_numpy(dtype=float)

# Normalize inputs — R's scale(): z-score per column, sd with ddof=1.
# NOTE (preserved from the original): train and test are scaled independently,
# each on its own mean/sd.
def scale(m):
    return (m - m.mean(axis=0)) / m.std(axis=0, ddof=1)

X_train_a = scale(X_train_a)
X_test_a = scale(X_test_a)

# ======================
# 3. Build DNN Model
# ======================
def build_and_train_model(X_train, y_train, X_test, y_test):
    model = keras.Sequential([
        keras.Input(shape=(X_train.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        loss="categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],
    )

    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=64,
        validation_split=0.2,
        verbose=1,
    )

    # Predict on test set
    probs = model.predict(X_test)
    pred_classes = probs.argmax(axis=1)  # zero-indexed to match label
    acc = float((pred_classes == test_label).mean())

    return {"model": model, "history": history, "accuracy": acc}

# ======================
# 4. Train and Evaluate
# ======================
nn_result_a = build_and_train_model(X_train_a, y_train_cat, X_test_a, y_test_cat)

print("✅ Neural Net - Pipeline A Accuracy:", round(nn_result_a["accuracy"], 4))

# PIPELINE B
# ======================
# 1. Load Libraries
# ======================
# (already imported above)

# ======================
# 2. Prepare Data for Pipeline B
# ======================

# Same target transformation as Pipeline A
test_label = test_data["predicted_class"].cat.codes.to_numpy()
train_label = train_data["predicted_class"].cat.codes.to_numpy()

# One-hot encode target labels
y_train_cat = keras.utils.to_categorical(train_label)
y_test_cat = keras.utils.to_categorical(test_label)
num_classes = y_train_cat.shape[1]

# Define Pipeline B features:
top_features_b = [
    "energy_consumption_current", "co2_per_m2", "energy_per_m2",
    "insulation_score", "fuel_bill_individual_est", "fuel_cost_ratio",
    "fuel_bill_local_avg_est", "elec_price_kwh", "gas_avg_fixed_cost_annual",
    "energy_per_room", "electricity_avg_fixed_cost_annual", "log_energy_per_room",
    "income_bhc_2024", "area_km", "population_density", "senior_population",
    "built_form_Mid-Terrace", "built_form_Semi-Detached",
]

# Subset and convert to matrix
X_train_b = train_data[top_features_b].to_numpy(dtype=float)
X_test_b = test_data[top_features_b].to_numpy(dtype=float)

# Normalize
X_train_b = scale(X_train_b)
X_test_b = scale(X_test_b)

# ======================
# 3. Build DNN Model
# ======================
# (redefined as in the original — identical architecture)
def build_and_train_model(X_train, y_train, X_test, y_test):
    model = keras.Sequential([
        keras.Input(shape=(X_train.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        loss="categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],
    )

    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=64,
        validation_split=0.2,
        verbose=1,
    )

    # Evaluate
    probs = model.predict(X_test)
    pred_classes = probs.argmax(axis=1)
    acc = float((pred_classes == test_label).mean())

    return {"model": model, "history": history, "accuracy": acc}

# ======================
# 4. Train and Evaluate
# ======================
nn_result_b = build_and_train_model(X_train_b, y_train_cat, X_test_b, y_test_cat)

print("✅ Neural Net - Pipeline B Accuracy:", round(nn_result_b["accuracy"], 4))


## Step 14 — Multinomial Logistic Regression Baseline (Pipeline A and B)

In [ ]:
# ===============================
# 1. Load libraries
# ===============================
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ===============================
# 2. Define Variables for Each Pipeline
# ===============================
# Define predictor variables
pipeline_a_vars = [c for c in train_data.columns if c != "predicted_class"]

pipeline_b_vars = [c for c in pipeline_a_vars if c != "current_energy_efficiency"]
pipeline_b_vars = list(dict.fromkeys(
    pipeline_b_vars + ["energy_consumption_current", "co2_per_m2", "energy_per_m2"]
))

# ===============================
# 3. Train Multinomial Logistic Model
# ===============================
# nnet::multinom = unregularised multinomial (softmax) logistic regression;
# penalty=None matches multinom's lack of regularisation, maxit = 500.
# (multinomial is the default for the lbfgs solver when n_classes > 2)

# Pipeline A
logit_train_a = train_data[pipeline_a_vars + ["predicted_class"]]
logit_test_a = test_data[pipeline_a_vars + ["predicted_class"]]

logit_model_a = LogisticRegression(penalty=None, max_iter=500, solver="lbfgs")
logit_model_a.fit(logit_train_a[pipeline_a_vars], logit_train_a["predicted_class"])
logit_pred_a = pd.Categorical(logit_model_a.predict(logit_test_a[pipeline_a_vars]), categories=label_map)

# Pipeline B
logit_train_b = train_data[pipeline_b_vars + ["predicted_class"]]
logit_test_b = test_data[pipeline_b_vars + ["predicted_class"]]

logit_model_b = LogisticRegression(penalty=None, max_iter=500, solver="lbfgs")
logit_model_b.fit(logit_train_b[pipeline_b_vars], logit_train_b["predicted_class"])
logit_pred_b = pd.Categorical(logit_model_b.predict(logit_test_b[pipeline_b_vars]), categories=label_map)

# ===============================
# 4. Evaluate Models
# ===============================
# Pipeline A evaluation
acc_logit_a = accuracy_score(logit_test_a["predicted_class"], logit_pred_a)
print("📊 Logistic Regression - Pipeline A Accuracy:", round(acc_logit_a, 4))

# Pipeline B evaluation
acc_logit_b = accuracy_score(logit_test_b["predicted_class"], logit_pred_b)
print("📊 Logistic Regression - Pipeline B Accuracy:", round(acc_logit_b, 4))


## Step 15 — Model Evaluation

In [ ]:
# model_evaluation_metrics.py

# =========================
# 📦 Load Packages
# =========================
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    roc_auc_score,
)

# =========================
# 📊 Evaluation Function
# =========================
def evaluate_model(pred, actual, model_name="Model", pipeline="A"):
    actual = pd.Categorical(np.asarray(actual), categories=label_map)
    pred = pd.Categorical(np.asarray(pred), categories=label_map)

    # Confusion Matrix (caret orientation: rows = Prediction, cols = Reference)
    cm = (
        pd.crosstab(
            pd.Series(pred, name="Prediction"),
            pd.Series(actual, name="Reference"),
        )
        .reindex(index=label_map, columns=label_map, fill_value=0)
    )
    print(cm)
    print(classification_report(actual, pred, zero_division=0))

    # Accuracy
    acc = accuracy_score(actual, pred)

    # Macro F1 (manual per-class loop preserved from the original; its fp/fn
    # are transposed vs. convention, which only swaps precision/recall — F1 is
    # invariant to that swap)
    f1_per_class = []
    for cls in label_map:
        tp = cm.loc[cls, cls]
        fp = cm[cls].sum() - tp
        fn = cm.loc[cls].sum() - tp
        precision = 0 if tp + fp == 0 else tp / (tp + fp)
        recall = 0 if tp + fn == 0 else tp / (tp + fn)
        f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        f1_per_class.append(f1)
    macro_f1 = float(np.mean(f1_per_class))

    # Sensitivity & Specificity (caret byClass: one-vs-rest per class, averaged)
    total = cm.to_numpy().sum()
    sens, spec = [], []
    for cls in label_map:
        tp = cm.loc[cls, cls]
        actual_pos = cm[cls].sum()      # reference == cls
        pred_pos = cm.loc[cls].sum()    # prediction == cls
        fp_ovr = pred_pos - tp
        actual_neg = total - actual_pos
        tn = actual_neg - fp_ovr
        sens.append(np.nan if actual_pos == 0 else tp / actual_pos)
        spec.append(np.nan if actual_neg == 0 else tn / actual_neg)
    sensitivity = float(np.nanmean(sens))
    specificity = float(np.nanmean(spec))
    kappa = cohen_kappa_score(actual, pred)

    # ROC AUC — pROC::multiclass.roc(actual_num, pred_num): mean AUC over all
    # class pairs, using the numeric predicted class as the score.
    # max(auc, 1-auc) approximates pROC's direction = "auto".
    pred_num = pd.Categorical(pred, categories=label_map).codes
    actual_num = pd.Categorical(actual, categories=label_map).codes
    try:
        aucs = []
        for i, j in combinations(np.unique(actual_num), 2):
            mask = np.isin(actual_num, [i, j])
            if len(np.unique(actual_num[mask])) < 2:
                continue
            auc_ij = roc_auc_score((actual_num[mask] == j).astype(int), pred_num[mask])
            aucs.append(max(auc_ij, 1 - auc_ij))
        roc_auc = float(np.mean(aucs)) if aucs else np.nan
    except ValueError:
        roc_auc = np.nan

    # Final Summary Output
    print("\n========================")
    print("📋", model_name, "- Pipeline", pipeline)
    print("========================")
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Sensitivity:", round(sensitivity, 4))
    print("Specificity:", round(specificity, 4))
    print("Kappa:", round(kappa, 4))
    print("Multiclass ROC AUC:", round(roc_auc, 4), "\n")

# (the original's `source("evaluation.R")` self-source line was a no-op artifact)

# ==============================
# 📦 Pipeline A: Evaluation Set-Up
# ==============================
true_class_a = test_data["predicted_class"]  # Ensure this is a factor

# ==============================
# 📦 1. XGBoost - Pipeline A
# ==============================
evaluate_model(
    pred_class_a,
    test_data["predicted_class"],
    model_name="XGBoost",
    pipeline="A",
)

# ==============================
# 🌲 2. Random Forest - Pipeline A
# ==============================
evaluate_model(
    rf_pred_a,
    rf_test_a["predicted_class"],
    model_name="Random Forest",
    pipeline="A",
)

# ==============================
# 📉 3. Logistic Regression - Pipeline A
# ==============================
evaluate_model(
    logit_pred_a,
    logit_test_a["predicted_class"],
    model_name="Logistic Regression",
    pipeline="A",
)

# ==============================
# 📦 1. XGBoost - Pipeline B
# ==============================
evaluate_model(
    pred_class_b,
    test_data["predicted_class"],
    model_name="XGBoost",
    pipeline="B",
)

# ==============================
# 🌲 2. Random Forest - Pipeline B
# ==============================
evaluate_model(
    rf_pred_b,
    rf_test_b["predicted_class"],
    model_name="Random Forest",
    pipeline="B",
)

# ==============================
# 📉 3. Logistic Regression - Pipeline B
# ==============================
evaluate_model(
    logit_pred_b,
    logit_test_b["predicted_class"],
    model_name="Logistic Regression",
    pipeline="B",
)
